# **I. Environment Setup on Google Colab**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install necessary libraries
!pip install torch==2.2.0 torchvision==0.17.0
!pip install retina-face
!pip install gbc-face==2025.1.0 albumentations==1.3.1
!pip install pytorch-lightning==2.2.0 torchmetrics==1.4.0
!pip install pandas scikit-learn opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# **II. Data Cleaning & Preprocessing & Alignment Pipeline**


### *1 - Renaming Identities*

In [ ]:
# import os

# # Base folder where your identities are stored
# base_path = "/content/drive/MyDrive/PFE/datasets/VGG"

# # Allowed image extensions
# image_extensions = ('.jpg', '.jpeg', '.png')

# # Dictionary to hold identity image counts
# identity_counts = {}

# # Step 1: Gather identity image counts
# for identity_name in os.listdir(base_path):
#     identity_path = os.path.join(base_path, identity_name)
#     if os.path.isdir(identity_path):
#         image_files = [
#             f for f in os.listdir(identity_path)
#             if f.lower().endswith(image_extensions)
#         ]
#         identity_counts[identity_name] = len(image_files)

# # Step 2: Sort identities by image count (descending)
# sorted_identities = sorted(identity_counts.items(), key=lambda x: x[1], reverse=True)

# # Step 3: Rename folders to 001, 002, ...
# for idx, (old_name, count) in enumerate(sorted_identities, start=1):
#     new_name = f"{idx:03d}"  # 3-digit padded format
#     old_path = os.path.join(base_path, old_name)
#     new_path = os.path.join(base_path, new_name)

#     if os.path.exists(new_path):
#         print(f"⚠️ Skipping rename: {old_name} → {new_name} (already exists!)")
#         continue

#     os.rename(old_path, new_path)
#     print(f"✅ Renamed: {old_name} → {new_name}")

# print(f"\n🎯 Total identities renamed: {len(sorted_identities)}")


### *1 - Test the working of Alignment with RetinaFace's Landmarks*

In [ ]:
# import os
# import cv2
# import numpy as np
# from retinaface import RetinaFace
# import matplotlib.pyplot as plt

# # Reference facial landmarks for alignment
# REFERENCE_FACIAL_POINTS = np.array([
#     [38.2946, 51.6963],
#     [73.5318, 51.5014],
#     [56.0252, 71.7366],
#     [41.5493, 92.3655],
#     [70.7299, 92.2041]
# ], dtype=np.float32)

# # Alignment function using estimateAffine2D
# def align_face(src_img, landmarks, image_size=(112, 112)):
#     src_pts = np.array([
#         landmarks["left_eye"],
#         landmarks["right_eye"],
#         landmarks["nose"],
#         landmarks["mouth_left"],
#         landmarks["mouth_right"]
#     ], dtype=np.float32)

#     tform, _ = cv2.estimateAffine2D(src_pts, REFERENCE_FACIAL_POINTS, method=cv2.LMEDS)
#     aligned_face = cv2.warpAffine(src_img, tform, image_size, borderValue=0.0)
#     return aligned_face, src_pts

# # Base paths
# input_base_path = "/content/drive/MyDrive/PFE/datasets/VGG_filtered"
# output_base_path = "/content/drive/MyDrive/PFE/datasets/aligned_faces_test"
# image_extensions = ('.jpg', '.jpeg', '.png')

# os.makedirs(output_base_path, exist_ok=True)

# # Limit to just 4 images for testing
# tested = 0
# max_test = 4

# # Walk through folders
# for dirpath, _, filenames in os.walk(input_base_path):
#     for filename in filenames:
#         if not filename.lower().endswith(image_extensions):
#             continue

#         if tested >= max_test:
#             break

#         image_path = os.path.join(dirpath, filename)
#         img = cv2.imread(image_path)
#         if img is None:
#             continue

#         faces = RetinaFace.detect_faces(image_path)
#         if not faces:
#             continue

#         for idx, face_info in enumerate(faces.values()):
#             landmarks = face_info.get("landmarks", None)
#             if landmarks is None:
#                 continue

#             aligned, src_pts = align_face(img, landmarks)

#             # Draw landmarks on original image
#             debug_img = img.copy()
#             for pt in src_pts:
#                 cv2.circle(debug_img, tuple(pt.astype(int)), 2, (0, 255, 0), -1)

#             # Save and show results
#             out_name = f"{os.path.splitext(filename)[0]}_face{idx}.jpg"
#             cv2.imwrite(os.path.join(output_base_path, out_name), aligned)

#             # Display both original and aligned images
#             plt.figure(figsize=(8, 4))
#             plt.subplot(1, 2, 1)
#             plt.imshow(cv2.cvtColor(debug_img, cv2.COLOR_BGR2RGB))
#             plt.title('Original + Landmarks')
#             plt.axis('off')

#             plt.subplot(1, 2, 2)
#             plt.imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB))
#             plt.title('Aligned Face')
#             plt.axis('off')
#             plt.suptitle(out_name)
#             plt.show()

#             tested += 1
#             break  # Test one face per image


### *2 - Face Detection Cleanup and Dataset Pruning Script*

In [ ]:
# import os
# import cv2
# import shutil
# from retinaface import RetinaFace

# # Base folder where your images are
# base_path = "/content/drive/MyDrive/PFE/datasets/VGG"

# # Allowed image extensions
# image_extensions = ('.jpg', '.jpeg', '.png')

# # Statistics counters
# total = 0
# detected = 0
# not_detected = 0

# # Keep track of remaining images per identity folder
# identity_image_count = {}

# # First Pass: Detect faces and delete undetectable images
# for dirpath, dirnames, filenames in os.walk(base_path):
#     for filename in filenames:
#         if filename.lower().endswith(image_extensions):
#             total += 1
#             image_path = os.path.join(dirpath, filename)

#             try:
#                 img = cv2.imread(image_path)
#                 if img is None:
#                     raise ValueError("Could not read image")

#                 faces = RetinaFace.detect_faces(image_path)

#                 if faces and len(faces) > 0:
#                     detected += 1
#                 else:
#                     # Delete image if no face detected
#                     os.remove(image_path)
#                     not_detected += 1

#             except Exception as e:
#                 # On error, also delete the image
#                 try:
#                     os.remove(image_path)
#                     not_detected += 1
#                     print(f"⚠️ Error and Deleted: {image_path} - {e}")
#                 except:
#                     print(f"❌ Could not delete corrupted image: {image_path}")

# # Second Pass: Delete identity folders with < 100 images
# for dirpath, dirnames, filenames in os.walk(base_path):
#     image_files = [f for f in filenames if f.lower().endswith(image_extensions)]
#     if len(image_files) < 100 and len(image_files) > 0:
#         try:
#             shutil.rmtree(dirpath)
#         except Exception as e:
#             print(f"⚠️ Failed to delete folder {dirpath}: {e}")

# print("\n=== SUMMARY ===")
# print(f"Total images processed : {total}")
# print(f"Faces detected         : {detected}")
# print(f"Images deleted         : {not_detected}")


### *3 - Calculating how much Identities Left*

In [ ]:
# import os

# # Base folder where your identities are stored
# base_path = "/content/drive/MyDrive/PFE/datasets/VGG"

# # Allowed image extensions
# image_extensions = ('.jpg', '.jpeg', '.png')

# # Dictionary to hold identity image counts
# identity_counts = {}

# # Walk through each identity folder
# for identity_name in os.listdir(base_path):
#     identity_path = os.path.join(base_path, identity_name)
#     if os.path.isdir(identity_path):
#         image_files = [
#             f for f in os.listdir(identity_path)
#             if f.lower().endswith(image_extensions)
#         ]
#         identity_counts[identity_name] = len(image_files)

# # Sort by image count (optional)
# identity_counts = dict(sorted(identity_counts.items(), key=lambda x: x[1], reverse=True))

# # Display results
# print(f"\n✅ Total identities remaining: {len(identity_counts)}\n")

# for identity, count in identity_counts.items():
#     print(f"🧍 Identity: {identity} — {count} images")


In [ ]:
# import os

# # Base path to renamed identity folders
# base_path = "/content/drive/MyDrive/PFE/datasets/VGG"

# # Allowed image extensions
# image_extensions = ('.jpg', '.jpeg', '.png')

# for identity in os.listdir(base_path):
#     identity_path = os.path.join(base_path, identity)

#     if os.path.isdir(identity_path):
#         # Get sorted list of image files
#         image_files = sorted([
#             f for f in os.listdir(identity_path)
#             if f.lower().endswith(image_extensions)
#         ])

#         # Rename each image to: <identity>_<image_number>.jpg
#         for idx, old_filename in enumerate(image_files, start=1):
#             ext = os.path.splitext(old_filename)[1].lower()
#             new_filename = f"{identity}_{idx:03d}{ext}"
#             old_path = os.path.join(identity_path, old_filename)
#             new_path = os.path.join(identity_path, new_filename)

#             os.rename(old_path, new_path)

# print("\n🎯 Finished renaming and ordering images.")


### *4 - Automated Batch Processing of Images in the VGG Dataset for Multi-Face Detection, Facial Landmark-Based Alignment Using RetinaFace, and Saving Aligned Faces Without Altering Originals into a Separate Filtered Directory*

In [ ]:
# import os
# import cv2
# import numpy as np
# from retinaface import RetinaFace

# REFERENCE_FACIAL_POINTS = np.array([
#     [38.2946, 51.6963],
#     [73.5318, 51.5014],
#     [56.0252, 71.7366],
#     [41.5493, 92.3655],
#     [70.7299, 92.2041]
# ], dtype=np.float32)

# def align_face(img, landmarks, image_size=(112, 112)):
#     src_pts = np.array([
#         landmarks["left_eye"],
#         landmarks["right_eye"],
#         landmarks["nose"],
#         landmarks["mouth_left"],
#         landmarks["mouth_right"]
#     ], dtype=np.float32)

#     tform, _ = cv2.estimateAffine2D(src_pts, REFERENCE_FACIAL_POINTS, method=cv2.LMEDS)
#     if tform is None:
#         return None

#     aligned = cv2.warpAffine(img, tform, image_size, borderValue=0.0)
#     return aligned

# # Paths
# input_base_path = "/content/drive/MyDrive/PFE/datasets/VGG"
# output_base_path = "/content/drive/MyDrive/PFE/datasets/vgg_filtered"
# log_path = "/content/drive/MyDrive/PFE/datasets/processed_folders.txt"

# image_extensions = ('.jpg', '.jpeg', '.png')
# max_folders_per_run = 15

# # Charger les dossiers déjà traités
# if os.path.exists(log_path):
#     with open(log_path, 'r') as f:
#         done_folders = set(line.strip() for line in f)
# else:
#     done_folders = set()

# processed_count = 0
# processed_folders = 0

# folder_names = sorted([f for f in os.listdir(input_base_path) if os.path.isdir(os.path.join(input_base_path, f))])

# for folder in folder_names:
#     if folder in done_folders:
#         continue

#     if processed_folders >= max_folders_per_run:
#         break

#     dirpath = os.path.join(input_base_path, folder)
#     output_dir = os.path.join(output_base_path, folder)
#     os.makedirs(output_dir, exist_ok=True)

#     filenames = os.listdir(dirpath)
#     print(f"📂 Processing folder: {folder} with {len(filenames)} files")

#     for i, filename in enumerate(filenames, 1):
#         if not filename.lower().endswith(image_extensions):
#             continue

#         image_path = os.path.join(dirpath, filename)
#         try:
#             img = cv2.imread(image_path)
#             if img is None:
#                 continue

#             faces = RetinaFace.detect_faces(image_path)
#             if not faces:
#                 continue

#             for idx, face_info in enumerate(faces.values()):
#                 landmarks = face_info.get("landmarks", None)
#                 if landmarks is None:
#                     continue

#                 aligned_face = align_face(img, landmarks)
#                 if aligned_face is None:
#                     continue

#                 out_filename = f"{os.path.splitext(filename)[0]}_face{idx}.jpg"
#                 out_path = os.path.join(output_dir, out_filename)
#                 cv2.imwrite(out_path, aligned_face)

#             processed_count += 1

#         except Exception as e:
#             print(f"❌ Error in {image_path}: {e}")
#             continue

#     # 📝 Ajouter le dossier au fichier de log
#     with open(log_path, 'a') as f:
#         f.write(folder + "\n")

#     processed_folders += 1

# print(f"\n🎉 Done. {processed_folders} new folders processed. {processed_count} images saved.")


### *3 landmarks logic : That worked perfect with my dataset*

In [ ]:
import os
import cv2
import numpy as np
import math
from retinaface import RetinaFace

# === Reference 3-point facial landmarks (eyes + nose) ===
REFERENCE_FACIAL_POINTS = np.array([
    [38.2946, 51.6963],  # left eye
    [73.5318, 51.5014],  # right eye
    [56.0252, 71.7366],  # nose
], dtype=np.float32)

def get_rotation_angle(landmarks):
    left_eye = np.array(landmarks["left_eye"])
    right_eye = np.array(landmarks["right_eye"])
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    return math.degrees(math.atan2(dy, dx))

def align_face(img, landmarks, image_size=(112, 112)):
    src_pts = np.array([
        landmarks["left_eye"],
        landmarks["right_eye"],
        landmarks["nose"]
    ], dtype=np.float32)
    tform = cv2.getAffineTransform(src_pts, REFERENCE_FACIAL_POINTS)
    aligned = cv2.warpAffine(img, tform, image_size, borderValue=0.0)
    return aligned

# === Paths and settings ===
input_base_path = "/content/drive/MyDrive/PFE/datasets/VGG"
output_base_path = "/content/drive/MyDrive/PFE/datasets/vgg_filtered"
log_path = "/content/drive/MyDrive/PFE/datasets/processed_folders.txt"

image_extensions = ('.jpg', '.jpeg', '.png')
max_folders_per_run = 15

# Load processed folders log
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        done_folders = set(line.strip() for line in f)
else:
    done_folders = set()

processed_count = 0
processed_folders = 0

folder_names = sorted([f for f in os.listdir(input_base_path) if os.path.isdir(os.path.join(input_base_path, f))])

for folder in folder_names:
    if folder in done_folders:
        continue

    if processed_folders >= max_folders_per_run:
        break

    dirpath = os.path.join(input_base_path, folder)
    output_dir = os.path.join(output_base_path, folder)
    os.makedirs(output_dir, exist_ok=True)

    filenames = [f for f in os.listdir(dirpath) if f.lower().endswith(image_extensions)]
    print(f"📂 Processing folder: {folder} with {len(filenames)} images")

    for filename in filenames:
        image_path = os.path.join(dirpath, filename)
        try:
            img = cv2.imread(image_path)
            if img is None:
                continue

            faces = RetinaFace.detect_faces(image_path)
            if not faces:
                continue

            for idx, face_info in enumerate(faces.values()):
                landmarks = face_info.get("landmarks")
                if landmarks is None:
                    continue

                angle = get_rotation_angle(landmarks)
                if abs(angle) > 5:
                    aligned_face = align_face(img, landmarks)
                else:
                    aligned_face = img

                out_filename = f"{os.path.splitext(filename)[0]}_face{idx}.jpg"
                out_path = os.path.join(output_dir, out_filename)
                cv2.imwrite(out_path, aligned_face)

            processed_count += 1

        except Exception as e:
            print(f"❌ Error in {image_path}: {e}")
            continue

    # Log folder as processed
    with open(log_path, 'a') as f:
        f.write(folder + "\n")

    processed_folders += 1

print(f"\n🎉 Done. {processed_folders} new folders processed. {processed_count} images saved.")


📂 Processing folder: 125 with 98 images


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5


25-06-11 22:23:41 - Directory /root/.deepface created
25-06-11 22:23:41 - Directory /root/.deepface/weights created
25-06-11 22:23:41 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


100%|██████████| 119M/119M [00:00<00:00, 327MB/s]


📂 Processing folder: 126 with 97 images
📂 Processing folder: 127 with 97 images
📂 Processing folder: 128 with 97 images
📂 Processing folder: 129 with 97 images
📂 Processing folder: 130 with 97 images
📂 Processing folder: 131 with 96 images
📂 Processing folder: 132 with 96 images
📂 Processing folder: 133 with 96 images
📂 Processing folder: 134 with 95 images
📂 Processing folder: 135 with 94 images
📂 Processing folder: 136 with 94 images
📂 Processing folder: 137 with 91 images
📂 Processing folder: 138 with 88 images
📂 Processing folder: 139 with 83 images

🎉 Done. 15 new folders processed. 1407 images saved.


### Script to process a single specified folder

5 landmarks :

In [ ]:
import os
import cv2
import numpy as np
from retinaface import RetinaFace
import math

REFERENCE_FACIAL_POINTS = np.array([
    [38.2946, 51.6963],
    [73.5318, 51.5014],
    [56.0252, 71.7366],
    [41.5493, 92.3655],
    [70.7299, 92.2041]
], dtype=np.float32)

def get_rotation_angle(landmarks):
    left_eye = np.array(landmarks["left_eye"])
    right_eye = np.array(landmarks["right_eye"])
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    return math.degrees(math.atan2(dy, dx))

def align_face(img, landmarks, image_size=(112, 112), padding=50):
    padded_img = cv2.copyMakeBorder(img, padding, padding, padding, padding, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    src_pts = np.array([
        landmarks["left_eye"],
        landmarks["right_eye"],
        landmarks["nose"],
        landmarks["mouth_left"],
        landmarks["mouth_right"]
    ], dtype=np.float32) + padding

    tform, _ = cv2.estimateAffine2D(src_pts, REFERENCE_FACIAL_POINTS, method=cv2.LMEDS)
    if tform is None:
        return None
    return cv2.warpAffine(padded_img, tform, image_size, borderValue=0.0)

# === Set your input/output paths ===
input_folder = "/content/drive/MyDrive/PFE/datasets/VGG/025"
output_folder = "/content/drive/MyDrive/PFE/datasets/vgg_filtered/025"
os.makedirs(output_folder, exist_ok=True)

image_extensions = ('.jpg', '.jpeg', '.png')

filenames = [f for f in os.listdir(input_folder) if f.lower().endswith(image_extensions)]

print(f"📂 Processing folder: {input_folder} ({len(filenames)} images)")

for filename in filenames:
    image_path = os.path.join(input_folder, filename)
    try:
        img = cv2.imread(image_path)
        if img is None:
            continue

        faces = RetinaFace.detect_faces(image_path)
        if not faces:
            continue

        for idx, face_info in enumerate(faces.values()):
            landmarks = face_info.get("landmarks")
            if landmarks is None:
                continue

            angle = get_rotation_angle(landmarks)
            aligned = img if abs(angle) < 5 else align_face(img, landmarks)
            if aligned is None:
                continue

            out_filename = f"{os.path.splitext(filename)[0]}_face{idx}.jpg"
            out_path = os.path.join(output_folder, out_filename)
            cv2.imwrite(out_path, aligned)

    except Exception as e:
        print(f"❌ Error with {image_path}: {e}")

print(f"\n✅ Done. Processed all images in {input_folder}.")


📂 Processing folder: /content/drive/MyDrive/PFE/datasets/VGG/025 (106 images)


KeyboardInterrupt: 

3 landmarks:

In [ ]:
import os
import cv2
import numpy as np
import math
from retinaface import RetinaFace

# === Reference 3-point facial landmarks (eyes + nose) ===
REFERENCE_FACIAL_POINTS = np.array([
    [38.2946, 51.6963],  # left eye
    [73.5318, 51.5014],  # right eye
    [56.0252, 71.7366],  # nose
], dtype=np.float32)

def get_rotation_angle(landmarks):
    left_eye = np.array(landmarks["left_eye"])
    right_eye = np.array(landmarks["right_eye"])
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    return math.degrees(math.atan2(dy, dx))

def align_face(img, landmarks, image_size=(112, 112)):
    src_pts = np.array([
        landmarks["left_eye"],
        landmarks["right_eye"],
        landmarks["nose"]
    ], dtype=np.float32)
    tform = cv2.getAffineTransform(src_pts, REFERENCE_FACIAL_POINTS)
    return cv2.warpAffine(img, tform, image_size, borderValue=0.0)

# === Input/output folder setup ===
input_folder = "/content/drive/MyDrive/PFE/datasets/VGG/025"
output_folder = "/content/drive/MyDrive/PFE/datasets/vgg_filtered/025"
os.makedirs(output_folder, exist_ok=True)

# === Image file filtering ===
image_extensions = ('.jpg', '.jpeg', '.png')
filenames = [f for f in os.listdir(input_folder) if f.lower().endswith(image_extensions)]

print(f"📂 Processing folder: {input_folder} ({len(filenames)} images)")

# === Loop through all images ===
for filename in filenames:
    image_path = os.path.join(input_folder, filename)
    try:
        img = cv2.imread(image_path)
        if img is None:
            continue

        faces = RetinaFace.detect_faces(image_path)
        if not faces:
            continue

        for idx, face_info in enumerate(faces.values()):
            landmarks = face_info.get("landmarks")
            if landmarks is None:
                continue

            angle = get_rotation_angle(landmarks)
            aligned = align_face(img, landmarks) if abs(angle) > 5 else img

            out_filename = f"{os.path.splitext(filename)[0]}_face{idx}.jpg"
            out_path = os.path.join(output_folder, out_filename)
            cv2.imwrite(out_path, aligned)

    except Exception as e:
        print(f"❌ Error with {image_path}: {e}")

print(f"\n✅ Done. All faces aligned and saved in: {output_folder}")


📂 Processing folder: /content/drive/MyDrive/PFE/datasets/VGG/025 (106 images)

✅ Done. All faces aligned and saved in: /content/drive/MyDrive/PFE/datasets/vgg_filtered/025


### Script to process just one image

In [ ]:
import os
import cv2
import numpy as np
import math
from retinaface import RetinaFace

# === Reference facial points (MUST match landmark order) ===
REFERENCE_FACIAL_POINTS = np.array([
    [38.2946, 51.6963],    # left eye
    [73.5318, 51.5014],    # right eye
    [56.0252, 71.7366],    # nose
], dtype=np.float32)

# === Compute rotation angle (optional, for logging/debugging) ===
def get_rotation_angle(landmarks):
    left_eye = np.array(landmarks["left_eye"])
    right_eye = np.array(landmarks["right_eye"])
    dx = right_eye[0] - left_eye[0]
    dy = right_eye[1] - left_eye[1]
    return math.degrees(math.atan2(dy, dx))

# === Align face using only 3 landmarks (more stable) ===
def align_face(img, landmarks, image_size=(112, 112)):
    src_pts = np.array([
        landmarks["left_eye"],
        landmarks["right_eye"],
        landmarks["nose"]
    ], dtype=np.float32)

    tform = cv2.getAffineTransform(src_pts, REFERENCE_FACIAL_POINTS)
    aligned = cv2.warpAffine(img, tform, image_size, borderValue=0.0)
    return aligned

# === Paths ===
image_path = "/content/drive/MyDrive/PFE/datasets/VGG/025/025_033.jpg"
output_dir = "/content/drive/MyDrive/PFE/datasets/vgg_filtered/025"
os.makedirs(output_dir, exist_ok=True)

# === Load image ===
img = cv2.imread(image_path)
if img is None:
    print("❌ Failed to read the image.")
    exit()

# === Detect faces ===
faces = RetinaFace.detect_faces(image_path)
if not faces:
    print("❌ No faces detected.")
    exit()

# === Process and align faces ===
filename = os.path.basename(image_path)
filename_no_ext = os.path.splitext(filename)[0]

for idx, face_info in enumerate(faces.values()):
    landmarks = face_info.get("landmarks")
    if landmarks is None:
        print(f"⚠️ No landmarks found for face {idx}")
        continue

    angle = get_rotation_angle(landmarks)
    print(f"📐 Face {idx} rotation angle: {angle:.2f}°")

    aligned = align_face(img, landmarks)

    if aligned is None:
        print(f"❌ Failed to align face {idx}")
        continue

    # === Save result ===
    out_filename = f"{filename_no_ext}_face{idx}.jpg"
    out_path = os.path.join(output_dir, out_filename)
    cv2.imwrite(out_path, aligned)
    print(f"✅ Face {idx} saved to {out_path}")


📐 Face 0 rotation angle: 177.61°
✅ Face 0 saved to /content/drive/MyDrive/PFE/datasets/vgg_filtered/025/025_033_face0.jpg


### Print Images That Detected More Than 1 Face :

In [ ]:
import os
from collections import defaultdict

# === Path to processed images (vgg_filtered) ===
filtered_dataset_path = "/content/drive/MyDrive/PFE/datasets/vgg_filtered"

# === Dictionary to track face counts per base image ===
face_counts = defaultdict(int)

# Traverse the filtered dataset
for folder in sorted(os.listdir(filtered_dataset_path)):
    folder_path = os.path.join(filtered_dataset_path, folder)
    if not os.path.isdir(folder_path):
        continue

    for filename in os.listdir(folder_path):
        if "_face" in filename:
            base_name = filename.split("_face")[0]  # e.g., 'img123'
            face_counts[os.path.join(folder, base_name)] += 1

# === Collect images with multiple faces ===
multi_face_images = [img for img, count in face_counts.items() if count > 1]

# === Output ===
print(f"\n🧑‍🤝‍🧑 Total images with more than 1 face: {len(multi_face_images)}")
for img in multi_face_images:
    print(f"➡️  {img}")

# Optional: Save to file
output_path = "images_with_multiple_faces.txt"
with open(output_path, "w") as f:
    for img in multi_face_images:
        f.write(img + "\n")

print(f"\n✅ List saved to: {output_path}")



🧑‍🤝‍🧑 Total images with more than 1 face: 0

✅ List saved to: images_with_multiple_faces.txt


### Count Images per Folder

In [ ]:
import os

# === Root directory containing subfolders like 001, 002, ..., nnn ===
root_dir = "/content/drive/MyDrive/PFE/datasets/vgg_filtered"

# === Image file extensions to count ===
image_extensions = ('.jpg', '.jpeg', '.png')

# === Loop through each folder and count images ===
folder_counts = {}

for folder in sorted(os.listdir(root_dir)):
    folder_path = os.path.join(root_dir, folder)
    if os.path.isdir(folder_path):
        image_files = [
            f for f in os.listdir(folder_path)
            if f.lower().endswith(image_extensions)
        ]
        folder_counts[folder] = len(image_files)

# === Print the results ===
print("📊 Image count per folder:\n")
for folder, count in folder_counts.items():
    print(f"{folder}: {count} images")

# === Total images across all folders ===
total_images = sum(folder_counts.values())
print(f"\n🧮 Total images: {total_images}")


### *9 - Data Splitting*

In [ ]:
# You already did this part in the Enhanced Work
# And here you want just to use the Test Set in the Splitted Data in Enhanced Work

Dataset splitting completed!
